In [34]:
import math


def norm_cdf(x: float) -> float:
    """
    Standard normal cumulative distribution function.
    """
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def bs_d1_d2(S: float, K: float, r: float, T: float, sigma: float):
    """
    Calculate d1 and d2 for Black-Scholes (no dividends).
    """

    if S <= 0 or K <= 0 or T <= 0 or sigma <= 0:
        raise ValueError("S, K, T, and sigma must be positive.")

    vol_sqrt_t = sigma * math.sqrt(T)

    # d1: measures relative position of spot vs strike
    d1 = (math.log(S / K) + (r + 0.5 * sigma * sigma) * T) / vol_sqrt_t

    # d2: adjusted threshold at expiry
    d2 = d1 - vol_sqrt_t

    return d1, d2


def bs_price(S: float, K: float, r: float, T: float, sigma: float, option_type="call"):
    """
    Black-Scholes price for European call/put (no dividends).
    """

    d1, d2 = bs_d1_d2(S, K, r, T, sigma)

    disc_r = math.exp(-r * T)

    if option_type.lower() == "call":
        # Call formula (no dividend)
        return S * norm_cdf(d1) - K * disc_r * norm_cdf(d2)

    elif option_type.lower() == "put":
        # Put formula (no dividend)
        return K * disc_r * norm_cdf(-d2) - S * norm_cdf(-d1)

    else:
        raise ValueError("option_type must be 'call' or 'put'.")

In [37]:
# =========================
# Validation against Excel ("Option" sheet)
# =========================

# Input parameters taken from the Excel "Option" sheet.
# These correspond to the base case used in the workbook.
S = 19.0       # Spot price
K = 17.0       # Strike price
T = 0.460  # Time to maturity (in years)
r = 0.005      # Risk-free interest rate
sigma = 0.3    # Volatility

# Compute option prices using the implemented Black-Scholes model.
call = bs_price(S, K, r, T, sigma, "call")
put = bs_price(S, K, r, T, sigma, "put")

# Display results for manual inspection.
print("Call price (Python):", call)
print("Put price (Python):", put)

# Expected values from Excel "Option" sheet.
# These values are used as reference for validation.
excel_call = 2.70
excel_put = 0.66


# Print comparison
print("=== Validation against Excel ===")
print(f"Call (Python): {call:.6f}")
print(f"Call (Excel) : {excel_call:.6f}")
print(f"Difference   : {call - excel_call:.6f}")
print()

print(f"Put (Python): {put:.6f}")
print(f"Put (Excel) : {excel_put:.6f}")
print(f"Difference  : {put - excel_put:.6f}")

Call price (Python): 2.696564799408952
Put price (Python): 0.6575097299555992
=== Validation against Excel ===
Call (Python): 2.696565
Call (Excel) : 2.700000
Difference   : -0.003435

Put (Python): 0.657510
Put (Excel) : 0.660000
Difference  : -0.002490


In [40]:
# Test cases for call options using the same spot price.
cases = [
    ("ITM", 15.0),  # Strike below spot: call is in the money
    ("ATM", S),  # Strike approximately equal to spot
    ("OTM", 23.0),  # Strike above spot: call is out of the money
]

for label, test_strike in cases:
    call_test = bs_price(S, test_strike, r, T, sigma, "call")
    put_test = bs_price(S, test_strike, r, T, sigma, "put")

    print(label)
    print("Call:", call_test)
    print("Put:", put_test)

    # Option prices should never be negative.
    assert call_test >= 0
    assert put_test >= 0

ITM
Call: 4.237113961444727
Put: 0.2026536060447084
ATM
Call: 1.55978012575879
Put: 1.516130342252099
OTM
Call: 0.40417330605046065
Put: 4.351334094437101


In [43]:
# =========================
# Test cases: ITM / ATM / OTM with expected behaviour
# =========================

test_cases = [
    ("ITM", 15.0, "Call option should have higher value due to intrinsic value."),
    ("ATM", S, "Call and put values should be moderate since spot is close to strike."),
    ("OTM", 23.0, "Call option should have lower value because strike is above spot."),
]

print("=== ITM / ATM / OTM Test Cases ===")

for label, test_strike, expectation in test_cases:
    call_test = bs_price(S=S, K=test_strike, r=r, T=T, sigma=sigma, option_type="call")
    put_test = bs_price(S=S, K=test_strike, r=r, T=T, sigma=sigma, option_type="put")

    print(f"\nScenario: {label}")
    print(f"Spot Price : {S:.2f}")
    print(f"Strike     : {test_strike:.2f}")
    print(f"Call Price : {call_test:.6f}")
    print(f"Put Price  : {put_test:.6f}")
    print(f"Expected   : {expectation}")

=== ITM / ATM / OTM Test Cases ===

Scenario: ITM
Spot Price : 19.00
Strike     : 15.00
Call Price : 4.237114
Put Price  : 0.202654
Expected   : Call option should have higher value due to intrinsic value.

Scenario: ATM
Spot Price : 19.00
Strike     : 19.00
Call Price : 1.559780
Put Price  : 1.516130
Expected   : Call and put values should be moderate since spot is close to strike.

Scenario: OTM
Spot Price : 19.00
Strike     : 23.00
Call Price : 0.404173
Put Price  : 4.351334
Expected   : Call option should have lower value because strike is above spot.
